Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan katalog menu mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

A. Membangun Katalog (Create) (bobot 20%)

Buat database baru bernama kopi_nusantara, collection bernama menu. Sisipkan minimal 12 dokumen menu dengan field bervariasi sesuai kategorinya:

    Menu kategori "Kopi": wajib memiliki field nama_menu, harga, kategori, dan tingkat_kepahitan (skala 1-5).
    Menu kategori "Makanan": wajib memiliki field nama_menu, harga, kategori, dan pedas_level (skala 0-3).
    Boleh menambahkan field lain sesuai kreativitas anda (mis. stok, rekomendasi_barista).

In [1]:
from pymongo import MongoClient

# Hubungkan ke instance lokal MongoDB.
client = MongoClient("mongodb://localhost:27017/")
db_kopi = client["kopi_nusantara"]
koleksi_menu = db_kopi["menu"]

# Bersihkan data lama jika cell dijalankan ulang.
koleksi_menu.delete_many({})

# Sisipkan 12 dokumen: 6 menu Kopi dan 6 menu Makanan.
koleksi_menu.insert_many([
    {"nama_menu": "Espresso", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 40},
    {"nama_menu": "Americano", "kategori": "Kopi", "harga": 20000, "tingkat_kepahitan": 4, "stok": 50},
    {"nama_menu": "Cappuccino", "kategori": "Kopi", "harga": 25000, "tingkat_kepahitan": 3, "stok": 35},
    {"nama_menu": "Cafe Latte", "kategori": "Kopi", "harga": 25000, "tingkat_kepahitan": 2, "stok": 30},
    {"nama_menu": "Kopi Tubruk", "kategori": "Kopi", "harga": 15000, "tingkat_kepahitan": 5, "stok": 60},
    {"nama_menu": "Caramel Macchiato", "kategori": "Kopi", "harga": 30000, "tingkat_kepahitan": 1, "stok": 25},
    {"nama_menu": "Nasi Goreng Spesial", "kategori": "Makanan", "harga": 35000, "pedas_level": 3, "stok": 20},
    {"nama_menu": "Mie Goreng Rempah", "kategori": "Makanan", "harga": 25000, "pedas_level": 2, "stok": 25},
    {"nama_menu": "Kentang Goreng", "kategori": "Makanan", "harga": 20000, "pedas_level": 0, "stok": 45},
    {"nama_menu": "Roti Bakar Cokelat", "kategori": "Makanan", "harga": 18000, "pedas_level": 0, "stok": 30},
    {"nama_menu": "Ayam Geprek Sambal Bawang", "kategori": "Makanan", "harga": 28000, "pedas_level": 3, "stok": 20},
    {"nama_menu": "Singkong Goreng Keju", "kategori": "Makanan", "harga": 15000, "pedas_level": 0, "stok": 40}
])

print(f"Katalog menu berhasil disimpan. Total: {koleksi_menu.count_documents({})} dokumen.")

Katalog menu berhasil disimpan. Total: 12 dokumen.


B. Kueri Bertingkat (Read) (bobot 20%)

Tampilkan hasil dari tiga kueri berikut :

    Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3.
    Seluruh menu dengan harga di antara 15000 dan 30000.
    Menu yang tidak memiliki field pedas_level sama sekali.

In [2]:
# 1. Menu Kopi dengan tingkat_kepahitan > 3.
print("=== 1. Menu Kopi dengan Kepahitan > 3 ===")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {m['nama_menu']} (Kepahitan: {m['tingkat_kepahitan']})")

# 2. Menu dengan harga antara 15.000 dan 30.000.
print("\n=== 2. Menu dengan Harga Antara 15.000 - 30.000 ===")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {m['nama_menu']} (Rp{m['harga']:,})")

# 3. Menu yang tidak memiliki field pedas_level sama sekali.
print("\n=== 3. Menu Tanpa Field 'pedas_level' ===")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {m['nama_menu']} ({m['kategori']})")

=== 1. Menu Kopi dengan Kepahitan > 3 ===
  - Espresso (Kepahitan: 5)
  - Americano (Kepahitan: 4)
  - Kopi Tubruk (Kepahitan: 5)

=== 2. Menu dengan Harga Antara 15.000 - 30.000 ===
  - Espresso (Rp18,000)
  - Americano (Rp20,000)
  - Cappuccino (Rp25,000)
  - Cafe Latte (Rp25,000)
  - Kopi Tubruk (Rp15,000)
  - Caramel Macchiato (Rp30,000)
  - Mie Goreng Rempah (Rp25,000)
  - Kentang Goreng (Rp20,000)
  - Roti Bakar Cokelat (Rp18,000)
  - Ayam Geprek Sambal Bawang (Rp28,000)
  - Singkong Goreng Keju (Rp15,000)

=== 3. Menu Tanpa Field 'pedas_level' ===
  - Espresso (Kopi)
  - Americano (Kopi)
  - Cappuccino (Kopi)
  - Cafe Latte (Kopi)
  - Kopi Tubruk (Kopi)
  - Caramel Macchiato (Kopi)


C. Pembaruan Data (Update) (bobot 15%)

Tambahkan field status bernilai "Tersedia" ke seluruh dokumen menggunakan update_many(). Kemudian, ubah status menjadi "Habis" khusus untuk 2 menu pilihan anda menggunakan update_one().

In [3]:
# Tambahkan field status = 'Tersedia' ke semua dokumen.
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Ubah status 2 menu tertentu menjadi 'Habis'.
koleksi_menu.update_one({"nama_menu": "Americano"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Mie Goreng Rempah"}, {"$set": {"status": "Habis"}})

# Tampilkan hasil verifikasi.
print("Verifikasi status menu:")
for m in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {m['nama_menu']}: {m['status']}")

Verifikasi status menu:
  - Espresso: Tersedia
  - Americano: Habis
  - Cappuccino: Tersedia
  - Cafe Latte: Tersedia
  - Kopi Tubruk: Tersedia
  - Caramel Macchiato: Tersedia
  - Nasi Goreng Spesial: Tersedia
  - Mie Goreng Rempah: Habis
  - Kentang Goreng: Tersedia
  - Roti Bakar Cokelat: Tersedia
  - Ayam Geprek Sambal Bawang: Tersedia
  - Singkong Goreng Keju: Tersedia


D. Integrasi ke Spark (bobot 25%)

Ambil seluruh data dari collection menu, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/null pada sebagian dokumen, seperti tingkat_kepahitan yang hanya dimiliki menu Kopi). Tampilkan:

    printSchema() hasil konversi.
    Rata-rata harga per kategori menggunakan groupBy.

In [4]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round

# Inisialisasi SparkSession.
spark = SparkSession.builder \
    .appName("Tugas9_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Ambil data dari MongoDB dan ubah ke pandas DataFrame.
dokumen_menu = list(koleksi_menu.find({}))
df_pandas = pd.DataFrame(dokumen_menu)

# Konversi ObjectId agar terbaca rapi oleh Spark.
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Konversi ke Spark DataFrame.
df_spark_menu = spark.createDataFrame(df_pandas)

# 1. Tampilkan skema hasil konversi.
print("=== 1. SKEMA SPARK DATAFRAME ===")
df_spark_menu.printSchema()

# 2. Rata-rata harga per kategori menggunakan groupBy.
print("\n=== 2. RATA-RATA HARGA PER KATEGORI ===")
df_spark_menu.groupBy("kategori").agg(
    round(avg("harga"), 2).alias("rata_rata_harga")
).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/03 08:23:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== 1. SKEMA SPARK DATAFRAME ===
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)


=== 2. RATA-RATA HARGA PER KATEGORI ===


[Stage 0:>                                                        (0 + 12) / 12]

+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|       22166.67|
| Makanan|        23500.0|
+--------+---------------+



E. Kesimpulan (bobot 20%)

Tulis pada markdown cell (minimal 100 kata): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

Tantangan Integrasi Skema Fleksibel NoSQL ke Ekosistem Spark Tabular.

Tantangan utama saat mengonversi data dari MongoDB ke Spark DataFrame berakar pada perbedaan mendasar paradigma penyimpanan data. MongoDB bekerja dengan model dokumen tanpa skema kaku alias flexible schema, memungkinkan atribut seperti tingkat_kepahitan hanya muncul pada kategori Kopi dan pedas_level hanya ada pada kategori Makanan. Sebaliknya, Spark DataFrame menuntut struktur tabel relasional yang seragam di setiap barisnya.

Saat data dikonversi via Pandas ke Spark, Spark akan menggabungkan seluruh variasi field menjadi kolom tabel utuh. Dampaknya, dokumen yang tidak memiliki atribut tertentu otomatis terisi nilai kosong (NaN atau null). Selain itu, tipe data khusus BSON seperti ObjectId pada kolom _id tidak dapat dipetakan langsung oleh tipe data primitif Spark. Kendala ini diatasi dengan melakukan standarisasi data di layer perantara, yaitu mengubah _id menjadi format string biasa sebelum konversi, serta memahami bahwa Spark tetap dapat mengeksekusi operasi agregasi numerik (seperti avg dan groupBy) secara tepat karena Spark secara bawaan mengabaikan nilai null pada kolom kalkulasi tanpa memutus pipeline analitik.   